# Playfit Intelligence Lab — 01: Catalog Quality Audit

Este notebook analiza la calidad y cobertura del catálogo de juegos de Playfit.
La base de datos contiene **63,682 juegos** con datos de plataformas, tags, scores, ventas,
sentimiento de reseñas, matching externo y detección de duplicados.

**Objetivos:**
- Medir la cobertura real del catálogo (géneros, portadas, años, tags)
- Analizar la distribución del `data_confidence_score`
- Caracterizar los duplicados detectados y su estado de revisión
- Evaluar la calidad del matching externo (Metacritic, VGSales)
- Generar visualizaciones portfolio-grade

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from src.features.catalog_features import (
    load_games, load_signals, load_platforms, load_tags,
    load_duplicates, load_external_match,
    coverage_analysis, signal_quality_analysis,
    duplicate_analysis, external_match_analysis,
)

sns.set_theme(style="darkgrid", palette="viridis")
plt.rcParams['figure.figsize'] = (12, 6)
print("Librerías cargadas correctamente.")

## 1. Cobertura del Catálogo
Cargamos los datos y calculamos métricas generales de cobertura.

In [ ]:
games = load_games()
cov = coverage_analysis(games)
for k, v in cov.items():
    if isinstance(v, float):
        print(f"{k:30s}: {v:.2f}%")
    else:
        print(f"{k:30s}: {v}")
print(f"\nTotal juegos: {cov['total_games']:,}")

**Hallazgos típicos:**
- ~60% sin género asignado
- ~50% sin portada (cover_url)
- ~3-5% sin año de lanzamiento
- ~5% sin tags

## 2. Fuentes y Estados de Publicación

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

source = cov['source_distribution']
ax1.bar(source['source_type'], source['len'], color=['#2ecc71', '#3498db', '#e74c3c'])
ax1.set_title('Distribución por Source Type')
ax1.set_ylabel('Cantidad de juegos')

states = cov['release_state_distribution']
ax2.bar(states['release_state'], states['len'], color=['#9b59b6', '#f39c12'])
ax2.set_title('Distribución por Release State')
ax2.set_ylabel('Cantidad de juegos')

plt.tight_layout()
plt.savefig('../reports/figures/source_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Señales de Calidad (Data Confidence Score)
La vista `game_recommendation_enrichment_signals` expone un score de confianza (0-100) por juego.

In [ ]:
signals = load_signals()
sq = signal_quality_analysis(signals)
print("Distribución del Data Confidence Score:")
for k, v in sq.items():
    if isinstance(v, float):
        print(f"  {k:30s}: {v:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

confidence = signals['data_confidence_score'].to_numpy()
axes[0].hist(confidence, bins=50, color='#3498db', edgecolor='white')
axes[0].set_title('Distribución de Confidence Score')
axes[0].set_xlabel('Confidence Score')
axes[0].set_ylabel('Juegos')

has_scores = signals['best_critic_score'].is_not_null().sum()
no_scores = len(signals) - has_scores
axes[1].pie([has_scores, no_scores], labels=['Con scores', 'Sin scores'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
axes[1].set_title('Cobertura de Scores')

has_sales = (signals['has_sales'] == True).sum()
no_sales = len(signals) - has_sales
axes[2].pie([has_sales, no_sales], labels=['Con ventas', 'Sin ventas'],
            autopct='%1.1f%%', colors=['#3498db', '#f39c12'])
axes[2].set_title('Cobertura de Ventas')

plt.tight_layout()
plt.savefig('../reports/figures/quality_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Análisis de Duplicados

In [ ]:
da = duplicate_analysis()
print(f"Grupos de duplicados: {da['total_groups']}")
print(f"Candidatos afectados: {da['total_candidates']}")
print(f"Grupos con años diferentes: {da['groups_with_diff_years']}")
print(f"Max candidatos por grupo: {da['max_candidates_per_group']}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

status = da['status_distribution']
ax1.bar(status['status'], status['len'], color=['#2ecc71', '#f39c12', '#3498db', '#e74c3c', '#9b59b6', '#95a5a6'])
ax1.set_title('Estado de Grupos de Duplicados')
ax1.set_ylabel('Grupos')
ax1.tick_params(axis='x', rotation=45)

review = da['review_distribution']
ax2.bar(review['suggested_review'], review['len'], color=['#3498db', '#e74c3c', '#f39c12'])
ax2.set_title('Suggested Review')
ax2.set_ylabel('Grupos')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../reports/figures/duplicate_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Matching Externo (Metacritic + VGSales)

In [ ]:
matches = load_external_match()
ema = external_match_analysis(matches)
print(f"Total match candidates: {ema['total_matches']:,}")
print(f"Confianza promedio: {ema['avg_confidence']:.1f}/100")
print(f"Auto-aprobados (high confidence): {ema['high_confidence_approved']:,}")
print(f"Necesitan revisión: {ema['needs_review']:,}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

status_m = ema['status_distribution']
ax1.bar(status_m['status'], status_m['len'],
        color=['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#9b59b6'])
ax1.set_title('Estado de Match Candidates')
ax1.tick_params(axis='x', rotation=45)

source_m = ema['source_distribution']
ax2.bar(source_m['source'], source_m['len'], color=['#3498db', '#e74c3c'])
ax2.set_title('Distribución por Fuente')
ax2.set_ylabel('Candidatos')

plt.tight_layout()
plt.savefig('../reports/figures/external_matching.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Resumen y Conclusiones

**Métricas clave de calidad del catálogo:**

In [ ]:
print("=" * 60)
print("RESUMEN DE CALIDAD DEL CATÁLOGO PLAYFIT")
print("=" * 60)
print(f"Total juegos:                 {cov['total_games']:>8,}")
print(f"Sin género:                   {cov['no_genre']:>7.1f}%")
print(f"Sin portada:                  {cov['no_cover']:>7.1f}%")
print(f"Sin año:                      {cov['no_year']:>7.1f}%")
print(f"Confianza promedio:           {sq['avg_confidence']:>7.1f}/100")
print(f"Con scores:                   {sq['has_scores_pct']:>7.1f}%")
print(f"Con ventas:                   {sq['has_sales_pct']:>7.1f}%")
print(f"Con sentimiento:              {sq['has_sentiment_pct']:>7.1f}%")
print(f"Grupos duplicados:            {da['total_groups']:>8,}")
print(f"Match candidates:             {ema['total_matches']:>8,}")
print("=" * 60)

**Conclusiones:**
1. El catálogo tiene **masa crítica** (63K+ juegos) pero con **huecos significativos** de cobertura
2. El `data_confidence_score` permite filtrar recomendaciones según calidad de datos
3. Hay **914 grupos de duplicados** que requieren limpieza manual
4. El matching externo (Metacritic + VGSales) añade **35K+ candidatos** con confianza variable
5. Para el sistema de recomendación, usaremos el confidence score como penalización y los duplicados para evitar recomendar versiones redundantes